# TaskMarket comp normalizer

A two-stage pipeline for cleaning refurbished-electronics comp data so it can actually be grouped and priced against.

**Stage one** is deterministic normalization for the obvious stuff: storage units, screen sizes, carrier codes, color aliases.

**Stage two** is an LLM that matches the residual rows where naming variants are too far apart for string similarity. Calling the model on every row is wasteful. Calling it only on the residual is the trade you want.

## What it does

1. Loads a comp CSV, or generates a simulated one if none is supplied.
2. Normalizes storage, screen size, carrier, color with deterministic rules.
3. Similarity-matches the model string against the canonical catalog.
4. Routes anything below the similarity threshold to an LLM for semantic matching.
5. Writes three sheets to Excel: matched rows, unmatched rows for review, and a summary of where each match came from.

## Required input columns

Case-insensitive: `title`, `storage`, `condition`, `price`. Optional: `color`, `carrier`, `channel`, `sold_date`.

## How to use your own data

Set `INPUT_PATH` in the CONFIG cell. Set `ANTHROPIC_API_KEY` in your environment. If no API key is present the pipeline still runs but skips the LLM stage and routes residual rows straight to `needs_review`.

## Tunable parameters

All behavior controls live in the CONFIG cell below. The similarity threshold, the canonical catalog, the LLM model choice, the carrier and color alias maps. These are demonstration defaults. In a real engagement these get tuned to the specific category, channels, and SKU range a refurbisher actually sells.


## Setup

Install dependencies if running fresh. The `anthropic` SDK is optional. Skip it and the pipeline still runs without the LLM stage.


In [1]:
# Uncomment if running for the first time.
# !pip install pandas openpyxl anthropic

In [2]:
import os
import re
import json
import random
from difflib import SequenceMatcher
import pandas as pd

## Config

Demonstration defaults. Tune to the catalog in practice.


In [3]:
INPUT_PATH = None  # Set to a CSV path to use real data. None = simulated.
OUTPUT_PATH = "comp_normalizer_output.xlsx"

# Similarity threshold for stage one. Rows above this skip the LLM. Rows
# below get sent to the LLM for semantic matching.
SIMILARITY_THRESHOLD = 0.78

# LLM config. Skip the LLM stage entirely if no API key is set.
LLM_MODEL = "claude-haiku-4-5-20251001"
LLM_ENABLED = bool(os.environ.get("ANTHROPIC_API_KEY"))
LLM_BATCH_SIZE = 10  # Rows per LLM call. Keeps token usage reasonable.
LLM_CONFIDENCE_CUTOFF = 0.6  # Accept LLM match if confidence at or above this.

# Canonical catalog. In a real engagement this comes from the
# refurbisher's product master, not hardcoded.
CANONICAL_CATALOG = [
    "iPhone 13 Pro",
    "iPhone 13 Pro Max",
    "iPhone 14",
    "iPhone 14 Pro",
    "Galaxy S22",
    "Galaxy S22 Ultra",
    "Galaxy S23",
    "iPad Air",
    "iPad Pro 11",
    "iPad Pro 12.9",
    "MacBook Air 13",
    "MacBook Pro 14",
    "MacBook Pro 16",
    "Pixel 7",
    "Pixel 7 Pro",
]

STORAGE_PATTERNS = [
    (r"(\d+(?:\.\d+)?)\s*tb", lambda m: int(float(m.group(1)) * 1024)),
    (r"(\d+)\s*gb", lambda m: int(m.group(1))),
    (r"\.(\d+)", lambda m: int(m.group(1))),
    (r"^(\d{2,4})$", lambda m: int(m.group(1))),
]

CARRIER_MAP = {
    "unlocked": "Unlocked",
    "factory unlocked": "Unlocked",
    "gsm unlocked": "Unlocked",
    "network unlocked": "Unlocked",
    "network: none": "Unlocked",
    "att": "AT&T",
    "at&t": "AT&T",
    "verizon": "Verizon",
    "vzw": "Verizon",
    "tmobile": "T-Mobile",
    "t-mobile": "T-Mobile",
    "tmo": "T-Mobile",
    "sprint": "Sprint",
}

COLOR_MAP = {
    "space gray": "Space Gray",
    "space grey": "Space Gray",
    "spacegray": "Space Gray",
    "silver": "Silver",
    "gold": "Gold",
    "rose gold": "Rose Gold",
    "graphite": "Graphite",
    "midnight": "Midnight",
    "starlight": "Starlight",
    "phantom black": "Phantom Black",
    "phantom white": "Phantom White",
}

SCREEN_SIZE_PATTERN = re.compile(
    r"(\d{1,2}(?:\.\d)?)\s*(?:inch|in|\"|”|')",
    re.IGNORECASE,
)

## Deterministic normalization

The cheap, fast, auditable stage. Handles the bulk of rows.


In [4]:
def normalize_storage(raw):
    """Resolve any storage value to integer GB. Returns None if unparseable."""
    if pd.isna(raw):
        return None
    s = str(raw).strip().lower().replace(" ", "")
    for pattern, resolver in STORAGE_PATTERNS:
        match = re.search(pattern, s)
        if match:
            try:
                return resolver(match)
            except (ValueError, IndexError):
                continue
    return None


def normalize_carrier(raw):
    if pd.isna(raw) or str(raw).strip() == "":
        return "Unknown"
    return CARRIER_MAP.get(str(raw).strip().lower(), str(raw).strip())


def normalize_color(raw):
    if pd.isna(raw) or str(raw).strip() == "":
        return "Unknown"
    return COLOR_MAP.get(str(raw).strip().lower(), str(raw).strip().title())


def normalize_screen_size(title):
    """Pull screen size from a title string. Returns float or None."""
    if pd.isna(title):
        return None
    match = SCREEN_SIZE_PATTERN.search(str(title))
    if match:
        return float(match.group(1))
    return None


def clean_model_string(title):
    """Strip storage, screen size, manufacturer prefix, and noise from a title."""
    s = str(title).lower()
    s = re.sub(r"\b\d+\s*(?:gb|tb)\b", "", s)
    s = re.sub(r"\.\d+\s*(?:gb|tb)?", "", s)
    s = re.sub(r"\b\d{1,2}(?:\.\d)?\s*(?:inch|in|\"|”)\b", "", s)
    s = re.sub(r"\b(?:apple|samsung|google)\b", "", s)
    s = re.sub(r"[^a-z0-9\s]", " ", s)
    s = re.sub(r"\s+", " ", s).strip()
    return s


def similarity_match(cleaned_title, catalog):
    """Stage one match. Cheap string similarity against the catalog."""
    best = None
    best_score = 0.0
    cleaned = cleaned_title.lower()
    for entry in catalog:
        entry_lower = entry.lower()
        score = SequenceMatcher(None, cleaned, entry_lower).ratio()
        if entry_lower in cleaned or cleaned in entry_lower:
            score = max(score, 0.85)
        if score > best_score:
            best_score = score
            best = entry
    return best, best_score

## LLM matching

Stage two. Only sees the rows the similarity stage couldn't resolve. Batched to keep call count down.


In [5]:
def llm_match_batch(titles, catalog):
    """Call Claude on rows the similarity stage couldn't resolve. Returns a
    list of dicts: {title, match, confidence, reason}. If the API isn't
    available, returns matches of None with a reason."""
    if not LLM_ENABLED:
        return [{"title": t, "match": None, "confidence": 0.0,
                 "reason": "llm_disabled"} for t in titles]

    try:
        from anthropic import Anthropic
    except ImportError:
        return [{"title": t, "match": None, "confidence": 0.0,
                 "reason": "anthropic_sdk_not_installed"} for t in titles]

    client = Anthropic()
    catalog_str = "\n".join(f"- {entry}" for entry in catalog)
    titles_str = "\n".join(f"{i+1}. {t}" for i, t in enumerate(titles))

    prompt = f"""You are matching messy product listings from a refurbished electronics seller against a canonical product catalog. Listings may have inconsistent spacing, abbreviations, typos, manufacturer prefixes, and embedded specs like storage or screen size.

Canonical catalog:
{catalog_str}

Listings to match:
{titles_str}

For each listing, return the best canonical match, a confidence between 0 and 1, and a one-line reason. If no entry in the catalog is plausibly the same product, return null for the match.

Respond with JSON only, in this exact shape:
{{{{"matches": [{{{{"index": 1, "match": "iPhone 13 Pro", "confidence": 0.95, "reason": "explicit model name with abbreviation"}}}}, ...]}}}}"""

    try:
        response = client.messages.create(
            model=LLM_MODEL,
            max_tokens=2000,
            messages=[{"role": "user", "content": prompt}],
        )
        text = response.content[0].text.strip()
        text = re.sub(r"^```(?:json)?\s*", "", text)
        text = re.sub(r"\s*```$", "", text)
        parsed = json.loads(text)
        results = []
        for i, title in enumerate(titles):
            entry = next((m for m in parsed.get("matches", [])
                          if m.get("index") == i + 1), None)
            if entry:
                results.append({
                    "title": title,
                    "match": entry.get("match"),
                    "confidence": float(entry.get("confidence", 0.0)),
                    "reason": entry.get("reason", ""),
                })
            else:
                results.append({"title": title, "match": None,
                                "confidence": 0.0, "reason": "no_llm_response"})
        return results
    except Exception as e:
        return [{"title": t, "match": None, "confidence": 0.0,
                 "reason": f"llm_error: {type(e).__name__}"} for t in titles]

## Simulated data

Two hundred rows across five models, with the storage values, carrier codes, color names, and screen size formats deliberately inconsistent. A small share of rows have the storage value stripped from the title to simulate a misfiled export.


In [6]:
def generate_simulated_data(n=200, seed=42):
    random.seed(seed)

    title_templates = {
        "iPhone 13 Pro": [
            "Apple iPhone 13 Pro {storage}GB",
            "iPhone 13Pro {storage}",
            "iPhone 13 Pro ({storage} GB)",
            "IP13Pro {storage}GB",
            "iphone 13 pro {storage}gb unlocked",
            "Apple iPhone 13 Pro .{storage}",
        ],
        "Galaxy S22": [
            "Samsung Galaxy S22 {storage}GB",
            "Galaxy S22 {storage}",
            "SM-S901 {storage}GB",
            "samsung galaxy s 22 {storage}gb",
        ],
        "MacBook Pro 14": [
            'MacBook Pro 14" {storage}GB',
            "MacBook Pro 14-inch {storage}GB",
            "MBP 14in {storage}",
            "Apple MacBook Pro 14 inch {storage}gb",
            "Macbook Pro 14 {storage}GB",
        ],
        "iPad Air": [
            "iPad Air {storage}GB",
            "Apple iPad Air ({storage} GB)",
            "ipad air {storage}gb wifi",
        ],
        "Pixel 7": [
            "Google Pixel 7 {storage}GB",
            "Pixel 7 {storage}",
            "google pixel7 {storage}gb",
        ],
    }

    storage_options = [128, 256, 512, 1024]
    storage_display = {128: ["128", "128GB", "128 GB"],
                       256: ["256", "256GB", "256 GB", ".256"],
                       512: ["512", "512GB", "512 GB"],
                       1024: ["1TB", "1 TB", "1024GB"]}

    conditions = ["Excellent", "Very Good", "Good", "Fair",
                  "excellent", "VG", "Like New", "Mint"]
    carriers = ["Unlocked", "factory unlocked", "GSM unlocked",
                "AT&T", "Verizon", "tmobile", "", "network: none"]
    colors = ["Space Gray", "space grey", "Silver", "Gold",
              "Graphite", "Midnight", "Phantom Black", ""]
    channels = ["Back Market", "Amazon Renewed", "eBay", "Swappa", "Direct"]

    rows = []
    for _ in range(n):
        model = random.choice(list(title_templates.keys()))
        template = random.choice(title_templates[model])
        storage_int = random.choice(storage_options)
        storage_str = random.choice(storage_display[storage_int])
        title = template.format(storage=storage_str)

        if random.random() < 0.05:
            title = title.replace(str(storage_int), "")

        base_prices = {
            "iPhone 13 Pro": 550, "Galaxy S22": 380,
            "MacBook Pro 14": 1450, "iPad Air": 420, "Pixel 7": 320,
        }
        storage_mult = {128: 1.0, 256: 1.15, 512: 1.35, 1024: 1.6}
        price = base_prices[model] * storage_mult[storage_int]
        price *= random.uniform(0.85, 1.15)

        rows.append({
            "title": title,
            "storage": storage_str if random.random() > 0.3 else "",
            "condition": random.choice(conditions),
            "color": random.choice(colors),
            "carrier": random.choice(carriers),
            "channel": random.choice(channels),
            "price": round(price, 2),
        })

    return pd.DataFrame(rows)

In [7]:
# Peek at what the messy input looks like.
sample_df = generate_simulated_data(n=20, seed=7)
sample_df.head(15)

,title,storage,condition,color,carrier,channel,price
0,MacBook Pro 14-inch GBGB,,Excellent,Gold,Unlocked,Back Market,2543.61
1,Apple iPad Air (128 GB),128,Very Good,Gold,Unlocked,Direct,410.49
2,Pixel 7,,Good,space grey,Verizon,Direct,354.41
3,Samsung Galaxy S22 256 GBGB,256 GB,Fair,,,eBay,464.81
4,ipad air 1 TBgb wifi,1 TB,Fair,space grey,Verizon,Direct,731.35
5,Apple iPad Air (1 TB GB),1 TB,Good,Midnight,GSM unlocked,Swappa,585.96
6,iPad Air 128 GBGB,128 GB,VG,,network: none,Back Market,467.31
7,iPhone 13 Pro (1024GB GB),1024GB,Mint,Graphite,,eBay,764.02
8,IP13Pro 512GB,,excellent,Silver,AT&T,Swappa,741.10
9,Apple iPad Air (128 GB),128,Like New,Graphite,,eBay,426.23


## Pipeline

Glue the stages together. Stage one runs on everything. Stage two only sees the residual.


In [8]:
def run_pipeline(df=None, output_path=OUTPUT_PATH):
    if df is None:
        print("No input provided. Generating simulated comp data.")
        df = generate_simulated_data()

    df.columns = [c.lower() for c in df.columns]
    df = df.copy()

    # Deterministic field normalization.
    df["storage_gb"] = df["storage"].apply(normalize_storage)
    title_storage = df["title"].apply(normalize_storage)
    df["storage_gb"] = df["storage_gb"].fillna(title_storage)
    df["screen_size"] = df["title"].apply(normalize_screen_size)
    df["carrier_clean"] = df["carrier"].apply(normalize_carrier) if "carrier" in df.columns else "Unknown"
    df["color_clean"] = df["color"].apply(normalize_color) if "color" in df.columns else "Unknown"

    # Stage one: similarity matching.
    cleaned = df["title"].apply(clean_model_string)
    sim_matches = cleaned.apply(lambda x: similarity_match(x, CANONICAL_CATALOG))
    df["model_matched"] = sim_matches.apply(lambda x: x[0])
    df["match_score"] = sim_matches.apply(lambda x: round(x[1], 3))
    df["match_stage"] = df["match_score"].apply(
        lambda s: "similarity" if s >= SIMILARITY_THRESHOLD else "pending_llm"
    )
    df["match_reason"] = ""

    # Stage two: LLM matching for the residual.
    residual_idx = df.index[df["match_stage"] == "pending_llm"].tolist()
    print(f"Stage one: {len(df) - len(residual_idx)} matched by similarity.")
    print(f"Stage two: {len(residual_idx)} rows pending LLM match. "
          f"LLM enabled: {LLM_ENABLED}.")

    for batch_start in range(0, len(residual_idx), LLM_BATCH_SIZE):
        batch = residual_idx[batch_start:batch_start + LLM_BATCH_SIZE]
        titles = df.loc[batch, "title"].tolist()
        results = llm_match_batch(titles, CANONICAL_CATALOG)
        for idx, result in zip(batch, results):
            if result["match"] and result["confidence"] >= LLM_CONFIDENCE_CUTOFF:
                df.at[idx, "model_matched"] = result["match"]
                df.at[idx, "match_score"] = result["confidence"]
                df.at[idx, "match_stage"] = "llm"
                df.at[idx, "match_reason"] = result["reason"]
            else:
                df.at[idx, "match_stage"] = "unmatched"
                df.at[idx, "match_reason"] = result["reason"]

    matched = df[df["match_stage"].isin(["similarity", "llm"])].copy()
    unmatched = df[df["match_stage"] == "unmatched"].copy()

    summary = pd.DataFrame({
        "metric": [
            "Total rows",
            "Matched at similarity stage",
            "Matched at LLM stage",
            "Unmatched (needs review)",
            "Storage resolved",
            "Storage unresolved",
            "LLM enabled",
            "Similarity threshold",
        ],
        "value": [
            len(df),
            (df["match_stage"] == "similarity").sum(),
            (df["match_stage"] == "llm").sum(),
            (df["match_stage"] == "unmatched").sum(),
            df["storage_gb"].notna().sum(),
            df["storage_gb"].isna().sum(),
            LLM_ENABLED,
            SIMILARITY_THRESHOLD,
        ],
    })

    with pd.ExcelWriter(output_path) as writer:
        matched.to_excel(writer, sheet_name="matched", index=False)
        unmatched.to_excel(writer, sheet_name="needs_review", index=False)
        summary.to_excel(writer, sheet_name="summary", index=False)

    print(f"Done. Output written to {output_path}")
    return matched, unmatched, summary

## Run it

Runs on simulated data by default. To use your own, set `INPUT_PATH` in the CONFIG cell and rerun.


In [9]:
if INPUT_PATH:
    input_df = pd.read_csv(INPUT_PATH)
    matched, unmatched, summary = run_pipeline(input_df)
else:
    matched, unmatched, summary = run_pipeline()

summary

No input provided. Generating simulated comp data.
Stage one: 157 matched by similarity.
Stage two: 43 rows pending LLM match. LLM enabled: False.
Done. Output written to comp_normalizer_output.xlsx


,metric,value
0,Total rows,200
1,Matched at similarity stage,157
2,Matched at LLM stage,0
3,Unmatched (needs review),43
4,Storage resolved,196
5,Storage unresolved,4
6,LLM enabled,False
7,Similarity threshold,0.78


In [10]:
# Spot check the matched rows.
matched[["title", "model_matched", "storage_gb", "match_stage", "match_score"]].head(15)

,title,model_matched,storage_gb,match_stage,match_score
0,Apple iPhone 13 Pro 512GB,iPhone 13 Pro,512.0,similarity,1.000
1,iPhone 13Pro 256,iPhone 13 Pro,256.0,similarity,0.828
2,"MacBook Pro 14"" .256GB",MacBook Pro 14,256.0,similarity,1.000
3,iPhone 13 Pro (512 GB GB),iPhone 13 Pro,512.0,similarity,0.897
4,Macbook Pro 14 512 GBGB,MacBook Pro 14,512.0,similarity,0.850
5,Apple iPad Air (1024GB GB),iPad Air,1024.0,similarity,0.850
6,Galaxy S22 1 TB,Galaxy S22,1024.0,similarity,1.000
7,iPhone 13 Pro (1 TB GB),iPhone 13 Pro,1024.0,similarity,0.897
8,iPad Air 512GB,iPad Air,512.0,similarity,1.000
9,samsung galaxy s 22 128gb,Galaxy S22,22128.0,similarity,0.952


In [11]:
# Spot check what didn't match. These are the rows a human would review
# in a real engagement. The reason column hints at why each one failed.
unmatched[["title", "match_stage", "match_score", "match_reason"]].head(15)

,title,match_stage,match_score,match_reason
21,IP13Pro 512GBGB,unmatched,0.571,llm_disabled
24,google pixel7 512GBgb,unmatched,0.571,llm_disabled
32,SM-S901 1024GBGB,unmatched,0.333,llm_disabled
35,SM-S901 128GB,unmatched,0.286,llm_disabled
36,SM-S901 128 GBGB,unmatched,0.276,llm_disabled
42,samsung galaxy s 22 1 TBgb,unmatched,0.714,llm_disabled
45,Apple MacBook Pro 14 inch 128 GBgb,unmatched,0.765,llm_disabled
48,SM-S901 1 TBGB,unmatched,0.286,llm_disabled
49,google pixel7 128GBgb,unmatched,0.571,llm_disabled
51,IP13Pro 128 GBGB,unmatched,0.552,llm_disabled


## What this is and isn't

This is one slice of a real ingestion pipeline. A production version would add a proper canonical product master pulled from your system, handling for accessories and bundles, channel-specific parsers for the formats that don't fit the general case, caching so identical messy titles don't get sent to the LLM twice, an embedding pre-filter to narrow the candidate set before the LLM call, a feedback loop where human review decisions train the matcher, and versioning so historical comps can be rerun after the rules change.

Book a call: [taskmarket.org/book](https://taskmarket.org/book)
